# 03_chunking.ipynb

**Actividad de Grado MIA UC**  
**Sistema Inteligente de Vigilancia de Riesgos Documentales basado en RAG y LLMs**

Este notebook transforma documentos PDF/DOCX en fragmentos recuperables para un sistema RAG.

## Objetivos
- Leer el inventario documental generado en el Notebook 02.
- Extraer texto desde PDF y DOCX.
- Aplicar limpieza básica.
- Generar chunks con solapamiento.
- Construir estadísticas de control.
- Guardar `chunks.csv` y `chunks.parquet` para embeddings.


In [ ]:
# =====================================================
# 03_chunking.ipynb
# Actividad de Grado MIA UC
# Patricia Patiño
# =====================================================

!pip -q install pandas pymupdf python-docx pyarrow openpyxl

## 1. Importar librerías

In [ ]:
import re
import os
import fitz  # PyMuPDF
import pandas as pd
from pathlib import Path
from docx import Document
from google.colab import files

pd.set_option('display.max_colwidth', 120)

print('Librerías cargadas correctamente')

## 2. Cargar inventario y documentos

Sube el archivo `inventario_corpus.csv` y los documentos PDF/DOCX del corpus.

In [ ]:
print('Sube inventario_corpus.csv y todos los documentos PDF/DOCX del corpus')
uploaded = files.upload()

print('\nArchivos cargados:', len(uploaded))

## 3. Leer inventario

In [ ]:
inventario_path = Path('inventario_corpus.csv')

if not inventario_path.exists():
    raise FileNotFoundError('No se encontró inventario_corpus.csv. Debes subirlo junto con los documentos.')

inventario = pd.read_csv(inventario_path)

columnas_requeridas = {'doc_id', 'filename', 'extension', 'tipo_documento'}
faltantes = columnas_requeridas - set(inventario.columns)

if faltantes:
    raise ValueError(f'Faltan columnas en el inventario: {faltantes}')

display(inventario.head())
print('Total documentos en inventario:', len(inventario))

## 4. Funciones de extracción de texto

Se conserva el número de página en PDFs para mantener trazabilidad documental.

In [ ]:
def extraer_texto_pdf(path):
    """Extrae texto página por página desde un PDF."""
    paginas = []
    doc = fitz.open(path)

    for page_num, page in enumerate(doc, start=1):
        texto = page.get_text('text')
        paginas.append({
            'page': page_num,
            'text': texto
        })

    doc.close()
    return paginas


def extraer_texto_docx(path):
    """Extrae texto desde un documento DOCX."""
    documento = Document(path)
    parrafos = [p.text for p in documento.paragraphs if p.text.strip()]
    texto = '\n'.join(parrafos)

    return [{
        'page': 1,
        'text': texto
    }]


def limpiar_texto(texto):
    """Aplica limpieza básica manteniendo contenido semántico."""
    if texto is None:
        return ''

    texto = texto.replace('\x00', ' ')
    texto = re.sub(r'\s+', ' ', texto)
    texto = texto.strip()

    return texto

## 5. Función de chunking

La estrategia base usa chunks por palabras con solapamiento. Esta estrategia es simple, reproducible y adecuada para construir un baseline RAG.

In [ ]:
def crear_chunks_por_palabras(texto, chunk_size=450, overlap=75, min_words=40):
    """
    Divide un texto en chunks de tamaño fijo por palabras.

    Parámetros:
    - chunk_size: número máximo de palabras por chunk.
    - overlap: número de palabras compartidas entre chunks consecutivos.
    - min_words: mínimo de palabras para conservar un chunk.
    """
    palabras = texto.split()

    if len(palabras) == 0:
        return []

    chunks = []
    start = 0

    while start < len(palabras):
        end = start + chunk_size
        chunk_words = palabras[start:end]
        chunk = ' '.join(chunk_words).strip()

        if len(chunk_words) >= min_words:
            chunks.append(chunk)

        if end >= len(palabras):
            break

        start = end - overlap

    return chunks

## 6. Configuración experimental

Estos parámetros quedan explícitos para poder reportarlos en el informe y compararlos en experimentos posteriores.

In [ ]:
CHUNK_SIZE = 450
CHUNK_OVERLAP = 75
MIN_WORDS = 40

print('Configuración de chunking')
print('chunk_size:', CHUNK_SIZE)
print('overlap:', CHUNK_OVERLAP)
print('min_words:', MIN_WORDS)

## 7. Procesar documentos y generar chunks

In [ ]:
registros_chunks = []
errores = []

for _, row in inventario.iterrows():
    doc_id = row['doc_id']
    filename = row['filename']
    extension = str(row['extension']).lower()
    tipo_documento = row['tipo_documento']

    path = Path(filename)

    if not path.exists():
        errores.append({
            'doc_id': doc_id,
            'filename': filename,
            'error': 'Archivo no encontrado en la sesión de Colab'
        })
        continue

    try:
        if extension == '.pdf':
            paginas = extraer_texto_pdf(path)
        elif extension == '.docx':
            paginas = extraer_texto_docx(path)
        else:
            errores.append({
                'doc_id': doc_id,
                'filename': filename,
                'error': f'Extensión no soportada: {extension}'
            })
            continue

        chunk_counter = 1

        for pagina in paginas:
            page = pagina['page']
            texto_limpio = limpiar_texto(pagina['text'])

            chunks = crear_chunks_por_palabras(
                texto_limpio,
                chunk_size=CHUNK_SIZE,
                overlap=CHUNK_OVERLAP,
                min_words=MIN_WORDS
            )

            for chunk in chunks:
                registros_chunks.append({
                    'doc_id': doc_id,
                    'filename': filename,
                    'tipo_documento': tipo_documento,
                    'page': page,
                    'chunk_id': f'{doc_id}_CHUNK_{chunk_counter:03d}',
                    'chunk_text': chunk,
                    'chunk_size_words': len(chunk.split()),
                    'chunk_size_chars': len(chunk),
                    'chunking_strategy': 'fixed_words_overlap',
                    'chunk_size_param': CHUNK_SIZE,
                    'overlap_param': CHUNK_OVERLAP
                })

                chunk_counter += 1

    except Exception as e:
        errores.append({
            'doc_id': doc_id,
            'filename': filename,
            'error': str(e)
        })

chunks_df = pd.DataFrame(registros_chunks)
errores_df = pd.DataFrame(errores)

print('Total chunks generados:', len(chunks_df))
print('Documentos con error:', len(errores_df))

display(chunks_df.head())

if len(errores_df) > 0:
    display(errores_df)

## 8. Validaciones de calidad

Estas validaciones permiten detectar documentos sin texto, problemas de extracción o chunks demasiado pequeños.

In [ ]:
if chunks_df.empty:
    raise ValueError('No se generaron chunks. Revisa si los archivos fueron cargados correctamente.')

resumen_documentos = (
    chunks_df.groupby(['doc_id', 'filename', 'tipo_documento'])
    .agg(
        cantidad_chunks=('chunk_id', 'count'),
        total_palabras=('chunk_size_words', 'sum'),
        promedio_palabras_chunk=('chunk_size_words', 'mean')
    )
    .reset_index()
)

display(resumen_documentos.head())

print('Documentos procesados:', resumen_documentos['doc_id'].nunique())
print('Chunks totales:', len(chunks_df))
print('Promedio palabras por chunk:', round(chunks_df['chunk_size_words'].mean(), 2))

## 9. Estadísticas por tipo documental

In [ ]:
stats_tipo = (
    chunks_df.groupby('tipo_documento')
    .agg(
        documentos=('doc_id', 'nunique'),
        chunks=('chunk_id', 'count'),
        palabras_totales=('chunk_size_words', 'sum'),
        promedio_palabras_chunk=('chunk_size_words', 'mean')
    )
    .reset_index()
)

display(stats_tipo)
display(chunks_df['chunk_size_words'].describe())

## 10. Revisión manual de ejemplos

Esta celda permite inspeccionar muestras de chunks para verificar si el texto conserva sentido.

In [ ]:
muestra = chunks_df.sample(min(5, len(chunks_df)), random_state=42)

for _, row in muestra.iterrows():
    print('='*100)
    print('chunk_id:', row['chunk_id'])
    print('documento:', row['filename'])
    print('tipo:', row['tipo_documento'])
    print('página:', row['page'])
    print('-'*100)
    print(row['chunk_text'][:1000])
    print()

## 11. Guardar artefactos

Se generan archivos para las siguientes etapas del pipeline:
- `chunks.parquet`: entrada principal para embeddings.
- `chunks.csv`: versión legible para revisión.
- `resumen_chunking.csv`: estadísticas por documento.
- `errores_chunking.csv`: trazabilidad de problemas, si existen.

In [ ]:
chunks_df.to_parquet('chunks.parquet', index=False)
chunks_df.to_csv('chunks.csv', index=False, encoding='utf-8-sig')
resumen_documentos.to_csv('resumen_chunking.csv', index=False, encoding='utf-8-sig')

if len(errores_df) > 0:
    errores_df.to_csv('errores_chunking.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
print('- chunks.parquet')
print('- chunks.csv')
print('- resumen_chunking.csv')
if len(errores_df) > 0:
    print('- errores_chunking.csv')

## 12. Descargar artefactos

In [ ]:
files.download('chunks.parquet')
files.download('chunks.csv')
files.download('resumen_chunking.csv')

if len(errores_df) > 0:
    files.download('errores_chunking.csv')